In [1]:
import starfile
import os
from pathlib import Path
import pandas as pd
import numpy as np
import einops
import mrcfile
import typer
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torch
from torch_image_interpolation.image_interpolation_2d import insert_into_image_2d
from torch_grid_utils.fftfreq_grid import dft_center
from torch_grid_utils.coordinate_grid import coordinate_grid
from fidder.erase import erase_masked_region

/home/gouauxlab/andecath/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def circle(
    radius: float,
    image_shape: tuple[int, int] | int,
    center: tuple[float, float] | None = None,
    device: torch.device | None = None,
) -> torch.Tensor:
    if isinstance(image_shape, int):
        image_shape = (image_shape, image_shape)
    if center is None:
        center = dft_center(image_shape, rfft=False, fftshifted=True)
    distances = coordinate_grid(
        image_shape=image_shape,
        center=center,
        norm=True,
        device=device,
    )
    mask = torch.zeros_like(distances, dtype=torch.float32)
    mask[distances < radius] = 1
    return mask

In [3]:
def Rx(angles_degrees: np.ndarray) -> np.ndarray:
    """Affine matrix for a rotation around the X-axis."""
    angles_degrees = np.asarray(angles_degrees).reshape(-1)
    c = np.cos(np.deg2rad(angles_degrees))
    s = np.sin(np.deg2rad(angles_degrees))
    matrices = einops.repeat(
        np.eye(4), 'i j -> n i j', n=len(angles_degrees)
    )
    matrices[:, 1, 1] = c
    matrices[:, 1, 2] = -s
    matrices[:, 2, 1] = s
    matrices[:, 2, 2] = c
    return np.squeeze(matrices)


def Ry(angles_degrees: np.ndarray) -> np.ndarray:
    """Affine matrix for a rotation around the Y-axis."""
    angles_degrees = np.asarray(angles_degrees).reshape(-1)
    c = np.cos(np.deg2rad(angles_degrees))
    s = np.sin(np.deg2rad(angles_degrees))
    matrices = einops.repeat(
        np.eye(4), 'i j -> n i j', n=len(angles_degrees)
    )
    matrices[:, 0, 0] = c
    matrices[:, 0, 2] = s
    matrices[:, 2, 0] = -s
    matrices[:, 2, 2] = c
    return np.squeeze(matrices)


def Rz(angles_degrees: float) -> np.ndarray:
    """Affine matrix for a rotation around the Z-axis."""
    angle_degrees = np.asarray(angles_degrees).reshape(-1)
    c = np.cos(np.deg2rad(angle_degrees))
    s = np.sin(np.deg2rad(angle_degrees))
    matrices = einops.repeat(
        np.eye(4), 'i j -> n i j', n=len(angle_degrees)
    )
    matrices[:, 0, 0] = c
    matrices[:, 0, 1] = -s
    matrices[:, 1, 0] = s
    matrices[:, 1, 1] = c
    return np.squeeze(matrices)


def S(shifts: np.ndarray) -> np.ndarray:
    """Affine matrices for shifts.

    Shifts supplied can be 2D or 3D.
    """
    shifts = np.asarray(shifts, dtype=float)
    if shifts.shape[-1] == 2:
        shifts = promote_2d_to_3d(shifts)
    shifts = np.array(shifts).reshape((-1, 3))
    matrices = einops.repeat(np.eye(4), 'i j -> n i j', n=shifts.shape[0])
    matrices[:, 0:3, 3] = shifts
    return np.squeeze(matrices)

def SC(scale: float) -> np.ndarray:
    """Affine matrices for scaling."""
    scale = np.asarray(scale).reshape(-1)
    matrices = einops.repeat(np.eye(4), 'i j -> n i j', n=len(scale))
    matrices[:, 0, 0] = scale
    matrices[:, 1, 1] = scale
    matrices[:, 2, 2] = scale
    return np.squeeze(matrices)

def promote_2d_to_3d(shifts: np.ndarray) -> np.ndarray:
    """Promote 2D vectors to 3D with zeros in the last dimension."""
    shifts = np.asarray(shifts).reshape(-1, 2)
    shifts = np.c_[shifts, np.zeros(shifts.shape[0])]
    return np.squeeze(shifts)

def tilt_series_alignment_parameters_to_relion_projection_matrices(
        specimen_shifts: pd.DataFrame,
        euler_angles: pd.DataFrame,
        tilt_image_dimensions: np.ndarray,
        tomogram_dimensions: np.ndarray,
):
    """Generate affine matrices transforming points in 3D to 2D in tilt-images.

    Projection model:
    3D specimen is rotated about its center then translated such that the projection
    of points onto the XY-plane gives their position in a tilt-image.

    More specifically
    - 3D specimen is rotated about its center by
        - shifting the origin to the specimen center
        - rotated extrinsically about the Y-axis by the tilt angle
        - rotated extrinsically about the Z-axis by the in plane rotation angle
    - 3D specimen is translated to align coordinate system with tilt-image
        - move center-of-rotation of specimen to center of tilt-image
        - move center-of-rotation of specimen to rotation center in tilt-image

    Parameters
    ----------
    specimen_shifts: XY-shifts which align the projected specimen with tilt-images
    euler_angles: YZX intrinsic Euler angles which transform the specimen
    tilt_image_dimensions: XY-dimensions of tilt-series.
    tomogram_dimensions: size of tomogram in XYZ
    """
    tilt_image_center = tilt_image_dimensions / 2
    specimen_center = tomogram_dimensions / 2

    # Transformations, defined in order of application
    s0 = S(-specimen_center)  # put specimen center-of-rotation at the origin
    r0 = Rx(euler_angles['rlnTomoXTilt'])  # rotate specimen around X-axis
    r1 = Ry(euler_angles['rlnTomoYTilt'])  # rotate specimen around Y-axis
    r2 = Rz(euler_angles['rlnTomoZRot'])  # rotate specimen around Z-axis
    sc1 = SC(4)
    s1 = S(specimen_shifts)  # shift projected specimen in xy (camera) plane
    s2 = S(tilt_image_center )  # move specimen back into tilt-image coordinate system

    # compose matrices
    transformations = s2 @  s1 @ sc1 @ r2 @ r1 @ r0 @ s0
    return np.squeeze(transformations)

In [10]:
def erase_aunps():
    try:
        tomo_name = list(Path.cwd().glob("*.rawtlt"))[0].stem
    except IndexError:
        typer.echo("No rawtlt file found")
        raise typer.Exit()
    
    tiltseries_data = starfile.read(f"{tomo_name}_relion_tiltseries.star")

    aunp_data = starfile.read("aunps/aunp_tm_BP_active_zone_all.star")
    aunp_coordinates = aunp_data[["faCoordinateX","faCoordinateY","faCoordinateZ"]].values

    with mrcfile.open(f"{tomo_name}_full_rec_BP_3DCTF_BIN4.mrc", header_only=True) as mrc_file:
        tomo_dimensions = np.array([mrc_file.header.nx, mrc_file.header.ny, mrc_file.header.nz])
    coordinates_2d = []
    for idx, row in tiltseries_data.iterrows():
        with mrcfile.open(row["rlnMicrographName"]) as mrc_file:
            image_dimensions = np.array([mrc_file.header.nx, mrc_file.header.ny])
            img_tensor = torch.from_numpy(mrc_file.data).float()
        projection_matrix =  tilt_series_alignment_parameters_to_relion_projection_matrices(
            specimen_shifts=row[["rlnTomoXShiftAngst", "rlnTomoYShiftAngst"]] / 2.5,
            euler_angles=row,
            tilt_image_dimensions=image_dimensions,
            tomogram_dimensions=tomo_dimensions
        )
        projected_coordinates = np.dot(
            projection_matrix,
            np.c_[aunp_coordinates, np.ones(aunp_coordinates.shape[0])].T
        ).T[:,:2]
        mask = torch.zeros_like(img_tensor, dtype=torch.float32)
        circle_mask = circle(9, (20,20))
        mask, _ = insert_into_image_2d(
            values=torch.ones(len(projected_coordinates),dtype=torch.float32),
            image=mask,
            coordinates=torch.tensor(projected_coordinates[:,::-1].copy()),
        )
        mask = F.conv2d(mask[None,None],circle_mask[None,None],padding='same')[0,0]
        print(mask.shape)
        print(img_tensor.shape)
        #mrcfile.write(
        #    Path(row["rlnMicrographName"]).parent / (Path(row["rlnMicrographName"]).stem + "_mask.mrc"),
        #    mask[0,0].numpy(),
        #    overwrite=True,
        #)
        erased_image = erase_masked_region(image=img_tensor, mask=mask)
        mrcfile.write(
            Path(row["rlnMicrographName"]).parent / (Path(row["rlnMicrographName"]).stem + "_aunpremoved.mrc"),
            erased_image.numpy(),
            overwrite=True,
        )
        

In [11]:
#os.chdir('/goliath/processing/Gouaux/CJS/BestTomo/ProcessingCJS/tomograms/15F1_tomograms/TOP_TOMOS/20231026_HippAu_14/best_alignment')
#erase_aunps()

In [ ]:
with open("/goliath/processing/Gouaux/CJS/BestTomo/ProcessingCJS/tomograms/15F1_tomograms/TOP_TOMOS/tomo_for_sta.txt","r") as f:
    tomos_for_sta = f.readlines()

#tomos_for_sta = ["20240209_waffleHip_165"]
for tomo in tomos_for_sta:
    tomo = tomo.strip()
    print(tomo)
    os.chdir(f'/goliath/processing/Gouaux/CJS/BestTomo/ProcessingCJS/tomograms/15F1_tomograms/TOP_TOMOS/{tomo}/best_alignment/')
    erase_aunps()

20230512-EG18pos2_80


torch.Size([3370, 4746])
torch.Size([3370, 4746])


/home/gouauxlab/andecath/env/lib/python3.11/site-packages/fidder/erase/sparse_local_mean.py:56: UserWarning: 
The coefficients of the spline returned have been computed as the
minimal norm least-squares solution of a (numerically) rank deficient
system (deficiency=33). If deficiency is large, the results may be
inaccurate. Deficiency may strongly depend on the value of eps.
  background_model = LSQBivariateSpline(y, x, z, tx, ty)


torch.Size([3370, 4746])
torch.Size([3370, 4746])


/home/gouauxlab/andecath/env/lib/python3.11/site-packages/fidder/erase/sparse_local_mean.py:56: UserWarning: 
The coefficients of the spline returned have been computed as the
minimal norm least-squares solution of a (numerically) rank deficient
system (deficiency=33). If deficiency is large, the results may be
inaccurate. Deficiency may strongly depend on the value of eps.
  background_model = LSQBivariateSpline(y, x, z, tx, ty)


torch.Size([3370, 4746])
torch.Size([3370, 4746])


... (32 outputs left out)

/home/gouauxlab/andecath/env/lib/python3.11/site-packages/fidder/erase/sparse_local_mean.py:56: UserWarning: 
The coefficients of the spline returned have been computed as the
minimal norm least-squares solution of a (numerically) rank deficient
system (deficiency=33). If deficiency is large, the results may be
inaccurate. Deficiency may strongly depend on the value of eps.
  background_model = LSQBivariateSpline(y, x, z, tx, ty)


torch.Size([3370, 4746])
torch.Size([3370, 4746])


/home/gouauxlab/andecath/env/lib/python3.11/site-packages/fidder/erase/sparse_local_mean.py:56: UserWarning: 
The coefficients of the spline returned have been computed as the
minimal norm least-squares solution of a (numerically) rank deficient
system (deficiency=33). If deficiency is large, the results may be
inaccurate. Deficiency may strongly depend on the value of eps.
  background_model = LSQBivariateSpline(y, x, z, tx, ty)


torch.Size([3370, 4746])
torch.Size([3370, 4746])
